# 03 — Train Pilot Models

**Renamed in spirit from "load pretrained models"** — notebook 01 confirmed
neither upstream repo ships checkpoints, so this notebook trains them.

**Architecture: imported directly from source, not re-derived** —
`MixturePrediction` (which internally builds the exact `MLP` upstream uses)
is imported from the cloned repo unchanged. Only the *training loop* around
it is this project's own code; see `src/models/pilot_mixture_model.py`'s
module docstring for exactly why (short version: upstream's own runner is a
large Hydra-driven grid search over every method in the paper's comparison
table via a custom grid DSL — correctly isolating just "BASE" from that grid
without spelunking `uq/runner.py` in full is a bigger reverse-engineering
risk than writing an auditable, from-scratch, 40-line training loop around
their real model class).

**This run trains `mixture_size=1`** (single Gaussian) — matching the go/
no-go plan's first branch, since that's the configuration whose assumptions
actually align with Neural Regression Collapse theory (05, pending formula
confirmation). Re-running with `mixture_size=3` (the paper's main default)
later is a one-line change once 05's go/no-go passes.

**Expected runtime:** a few minutes total for all 12 pilot datasets on CPU;
faster still on the T4 (these are small models — a few hundred parameters
to a few thousand, not millions).
**GPU:** used if available (falls back to CPU automatically), but not
required — these models are tiny.

**Verified against source before writing this notebook** (see the module
docstring in `src/models/pilot_mixture_model.py` for exact file/line
references): hidden_sizes=[128,128,128], dropout=0.2, AdamW(lr=1e-3),
batch_size=512 (capped to split size for the smallest datasets — CPU has
only ~87 training rows after splitting), early stopping with patience=30
on validation NLL.


## Step 1 — Locate project, import helpers

In [ ]:
import sys
from pathlib import Path

if "PROJECT_ROOT" not in dir():
    _here = Path.cwd()
    for candidate in [_here, *_here.parents]:
        if (candidate / "src" / "utils" / "env_utils.py").exists():
            PROJECT_ROOT = candidate
            break
    else:
        raise FileNotFoundError("PROJECT_ROOT not found. Run 00_environment.ipynb first.")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.utils import env_utils
from src.models import pilot_mixture_model as pmm

EXTERNAL_DIR = PROJECT_ROOT / "external" / "quantile-recalibration-training"
if not EXTERNAL_DIR.exists():
    print("External repo not found -- cloning now (normally done by notebook 01).")
    env_utils.clone_or_pull_repo(
        repo_url="https://github.com/Vekteur/quantile-recalibration-training.git",
        dest=EXTERNAL_DIR, branch="main",
    )

MixturePrediction = pmm.import_mixture_prediction(PROJECT_ROOT)
pmm.set_mixture_prediction_cls(MixturePrediction)
print(f"PROJECT_ROOT = {PROJECT_ROOT}")
print(f"Using real MixturePrediction from: {MixturePrediction.__module__}")


## Step 2 — Device selection (T4-aware) & load prepared splits from `02`

In [ ]:
import torch
import pickle

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Training device: {DEVICE}")
if DEVICE == "cuda":
    gpu_check = env_utils.verify_gpu_matches_target("T4")
    print(f"GPU: {gpu_check['device_name']} (matches T4 target: {gpu_check['matches_target']})")

splits_path = (PATHS["outputs"] if "PATHS" in dir() else PROJECT_ROOT / "outputs") / "uci_pilot_splits.pkl"
if not splits_path.exists():
    raise FileNotFoundError(f"{splits_path} not found -- run 02_prepare_datasets.ipynb first.")

with open(splits_path, "rb") as f:
    saved = pickle.load(f)
all_splits = saved["splits"]
SEED = saved["seed"]
print(f"Loaded prepared splits for {len(all_splits)} datasets (seed={SEED}).")


## Step 3 — Train one BASE (`mixture_size=1`) model per dataset

Sequential, not parallel: these models are small enough that the overhead
of parallelizing across 12 tiny datasets isn't worth the added complexity
for a pilot. `nb_workers`-style parallelism (as upstream's `run.py` supports)
is a reasonable addition later if this scales beyond the pilot group.

In [ ]:
MIXTURE_SIZE = 1  # go/no-go branch; change to 3 for the paper's main default later

results = {}
for name, splits in all_splits.items():
    print(f"\n=== Training {name} (train={len(splits['train']['x'])}, val={len(splits['val']['x'])}) ===")
    result = pmm.train_pilot_model(
        splits, mixture_size=MIXTURE_SIZE, seed=SEED, device=DEVICE, verbose=False,
    )
    results[name] = result
    print(f"  best_epoch={result['best_epoch']}  best_val_nll={result['best_val_nll']:.4f}  "
          f"(ran {len(result['history']['val_nll'])} epochs before stopping)")


## Step 4 — Sanity checks before saving anything

Two cheap, independent correctness signals: no dataset should have a
non-finite best val NLL, and every model should have actually improved
over its own first-epoch NLL (a model that never improves likely has a
bug, not just "a hard dataset" — patience=30 gives it plenty of room).

In [ ]:
import math

problems = []
for name, result in results.items():
    first_nll = result["history"]["val_nll"][0]
    best_nll = result["best_val_nll"]
    if not math.isfinite(best_nll):
        problems.append(f"{name}: non-finite best_val_nll ({best_nll})")
    elif best_nll >= first_nll:
        problems.append(f"{name}: never improved past epoch 0 (first={first_nll:.4f}, best={best_nll:.4f})")

if problems:
    print("[warn] Potential issues:")
    for p in problems:
        print(f"  - {p}")
else:
    print(f"All {len(results)} models: finite best val NLL, all improved past their first epoch.")

print("\nSummary:")
for name, result in results.items():
    print(f"  {name:10s}  best_val_nll={result['best_val_nll']:8.4f}  best_epoch={result['best_epoch']:4d}")


## Step 5 — Save checkpoints for notebook 04

In [ ]:
CKPT_DIR = (PATHS["checkpoints"] if "PATHS" in dir() else PROJECT_ROOT / "checkpoints") / f"mixture_{MIXTURE_SIZE}"
CKPT_DIR.mkdir(parents=True, exist_ok=True)

for name, result in results.items():
    module = result["module"]
    torch.save(
        {
            "state_dict": module.model.state_dict(),
            "input_size": module.input_size,
            "mixture_size": module.mixture_size,
            "best_val_nll": result["best_val_nll"],
            "best_epoch": result["best_epoch"],
            "seed": SEED,
        },
        CKPT_DIR / f"{name}.pt",
    )
print(f"Saved {len(results)} checkpoints to {CKPT_DIR}")


## Next steps

**Next:** `04_extract_features.ipynb` — loads these checkpoints, runs a
forward pass over each dataset's calibration split, and saves the
penultimate-layer features (already implemented as
`PilotMixtureLitModule.penultimate_features` — written in `03`'s own
source module while the architecture was fresh in context) for `05` to
compute NRC geometry from, once its formula-confirmation gate clears.
